기본 필요 모델 호출

In [9]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "SamilPwC-AXNode-GenAI/PwC-Embedding_expr"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3686.81it/s]


In [23]:
import pandas as pd
import numpy as np
import json
from pathlib import Path


데이터 불러오기 : 짧은 책소개

In [15]:
df1 = pd.read_csv("../도서데이터/books.csv")
display(df1['description'].head())

0    공항, 크루즈, 박물관, 경찰서 등 유럽 곳곳에서 의문의 사건이 발생했다. 뭔가 미...
1    "한 문제만 더." 《머도쿠》는 이 짧은 말이 가장 잘 어울리는 추리 퍼즐북이다. ...
2    일본의 대표 예술가 제아미, 문제적 천재 다자이 오사무, 일본 근대 문학의 아버지 ...
3    크툴루의 부름은 H.P. 러브크래프트의 세계를 다루는 테이블 롤플레잉 게임이다. 크...
4    80만 회원님의 식단을 책임지고 있는 밥피티의 첫 다이어트 레시피북. 일대일 온라인...
Name: description, dtype: object

In [ ]:
#짧은 책 소개를 리스트로 변환
books_description = df1['description'].tolist()

#동시에 임베딩
books_description_embeddings = model.encode(books_description)

TypeError: 'tuple' object is not callable

In [22]:
display(books_description_embeddings.shape)

(3000, 1024)

기존 csv에 병합

In [24]:
#옆의 여렝 한칸 내부에 , 로 차원을 모두 넣는다.
df1["descriptionEmbedding"] = [
    json.dumps(embedding.tolist())
    for embedding in books_description_embeddings
]

#저장할 폴더 위치
save_dir = Path("../embedding_data")
#폴더가 없으면 생성, 이미 있으면 그냥 넘어감
save_dir.mkdir(parents=True, exist_ok=True)

# 저장 경로
save_path = save_dir / "books_embeddings.csv"

#csv로 저장
df1.to_csv(save_path, index=False)

중복 제거

In [26]:
# itemId 기준 중복 제거
df1 = df1.drop_duplicates(
    subset=["itemId"],
    keep="first"
).reset_index(drop=True)

print(df1.shape)

# CSV용 문자열로 저장했던 embedding을 다시 리스트로 변환
embeddings = df1["descriptionEmbedding"].apply(json.loads).tolist()

print(len(embeddings))
print(len(embeddings[0]))

(2558, 16)
2558
1024


ChromaDB생성

In [27]:
import chromadb

# Vector DB 저장 위치
vector_db_path = Path("../vector_db")

# 로컬 Chroma DB 생성
client = chromadb.PersistentClient(
    path=str(vector_db_path)
)

# books 컬렉션 생성 + 코사인 유사도 검색 기능
collection = client.get_or_create_collection(
    name="books",
    configuration={
        "hnsw": {
            "space": "cosine"
        }
    }
)

# ChromaDB에 책 데이터 저장
collection.upsert(
    ids=df1["itemId"].astype(str).tolist(),
    embeddings=embeddings,
    documents=df1["description"].fillna("").tolist(),
    metadatas=[
        {
            "title": str(row["title"]),
            "author": str(row["author"]),
            "publisher": str(row["publisher"]),
            "category": str(row["category_name"]),
            "pubDate": str(row["pubDate"]),
            "priceSales": int(row["priceSales"])
        }
        for _, row in df1.iterrows()
    ]
)

In [28]:
#확인해보기
# 저장된 책 개수 확인
print(collection.count())

# 일부 데이터 확인
print(collection.peek(3))

2558
{'ids': ['296236093', '395486703', '397217574'], 'embeddings': array([[ 0.03237326,  0.00618598, -0.03570672, ..., -0.00605244,
        -0.00210198, -0.03007776],
       [ 0.01641113, -0.02598785, -0.00387304, ..., -0.03394416,
         0.00691089, -0.01397121],
       [-0.0057603 , -0.03469588,  0.01004256, ..., -0.05657293,
        -0.01516718, -0.02516403]], shape=(3, 1024)), 'documents': ['공항, 크루즈, 박물관, 경찰서 등 유럽 곳곳에서 의문의 사건이 발생했다. 뭔가 미심쩍은 자살부터 불운한 사고, 기묘한 미스터리까지 각기 다른 12개의 사건 현장이 당신을 기다린다. 천재 크리에이터 모데스토 가르시아가 기획한 이 책은 독특한 창의성과 놀라운 상상력으로 독자들을 추리 소설의 주인공으로 만든다.', '"한 문제만 더." 《머도쿠》는 이 짧은 말이 가장 잘 어울리는 추리 퍼즐북이다. &lt;파이낸셜 타임스&gt;의 퍼즐 디자이너 마누엘 가란드가 스도쿠의 규칙, 논리 퍼즐의 추론, 배틀십의 공간 감각을 새롭게 결합해 만든 《머도쿠》는 독자를 총 80개의 살인 사건 속으로 초대한다.', '일본의 대표 예술가 제아미, 문제적 천재 다자이 오사무, 일본 근대 문학의 아버지 나쓰메 소세키의 명문장부터, 1990년대 일본 전체를 들썩이게 한 드라마 〈롱 베케이션〉의 명대사 등 일본 문학사와 문화에서 중요한 120개의 문장이 한 권에 담겼다.'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'title': '당신은 사건 현장에 

유사도 검색 테스트

In [30]:
query = "추리하면서 사건을 해결하는 책"

query_embedding = model.encode([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

#타이틀만 확인해보기
display(results['metadatas'][0])

[{'pubDate': '2025-12-23',
  'title': '방탈출 퍼즐북 : 정탐정과 미스터리 실종 사건 - 종이 위에 펼쳐지는 리얼 탈출 게임 어드벤처',
  'priceSales': 17100,
  'publisher': '한빛라이프',
  'category': '건강/취미',
  'author': '정주영'},
 {'title': '크라임 퍼즐 - 문장 속에 숨겨진 범인을 찾는 두뇌 게임 100',
  'category': '건강/취미',
  'author': 'G.T. Karber',
  'priceSales': 17550,
  'publisher': '중앙books(중앙북스)',
  'pubDate': '2023-08-11'},
 {'title': '머도쿠 - 두뇌를 자극하는 추리 퍼즐 80',
  'author': '마누엘 가란드',
  'category': '건강/취미',
  'priceSales': 20700,
  'publisher': '중앙books(중앙북스)',
  'pubDate': '2026-06-18'},
 {'category': '건강/취미',
  'publisher': '중앙books(중앙북스)',
  'pubDate': '2022-05-30',
  'title': '당신은 사건 현장에 있습니다 - 일러스트 한 장으로 즐기는 추리 게임',
  'priceSales': 25200,
  'author': '모데스토 가르시아'},
 {'priceSales': 16650,
  'category': '소설/시/희곡',
  'publisher': '반타',
  'author': '정해연',
  'title': '내가 죽였다',
  'pubDate': '2026-06-18'}]